# JD Resume Matcher End-to-End Local Test

This notebook runs the full local workflow: JD keyword extraction, resume keyword extraction, Phase 2 validation, bidirectional skill matching, and compact result reporting.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
from pathlib import Path

from matcher.skill_matcher import match_resume_to_jd
from nodes.keyword_extractor_graph import run_keywords_extractor
from rexpand_pyutils_file import read_file
from utils.llm import LLM_USE_CACHE

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


def env_flag(name, default):
    """Read boolean toggles the same way the Lambda worker does."""
    return os.getenv(name, default).strip().lower() in {"1", "true", "yes", "y", "on"}


def env_float(name, default):
    """Read float settings defensively so bad local env values do not crash setup."""
    try:
        return float(os.getenv(name, default).strip())
    except ValueError:
        return float(default)


MATCH_USE_EMBEDDINGS = env_flag("MATCH_USE_EMBEDDINGS", "true")
MATCH_EMBEDDING_THRESHOLD = env_float("MATCH_EMBEDDING_THRESHOLD", "0.72")


def save_json(path, payload):
    """Persist notebook artifacts so you can inspect exact states after a run."""
    Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("LLM model:", os.getenv("LLM_MODEL", "gpt-4.1-mini"))
print("LLM use cache:", LLM_USE_CACHE)
print("Matcher use embeddings:", MATCH_USE_EMBEDDINGS)
print("Matcher embedding threshold:", MATCH_EMBEDDING_THRESHOLD)

In [ ]:
# Keep this False for a quick smoke test. Set it to True if you want to run
# against the long fixture in examples/job_description.md.
USE_FULL_JOB_DESCRIPTION = False

if USE_FULL_JOB_DESCRIPTION:
    jd_text = read_file("examples/job_description.md")
else:
    jd_text = """
    We are hiring a backend software engineer. The candidate must have Python,
    SQL, REST API development, and AWS experience. Strong communication and
    ownership are required. Docker experience is preferred.
    """

resume_text = """
I built Python and SQL data pipelines, developed REST APIs, deployed services
on AWS Lambda, and collaborated with product teams. I have also used Docker for
local development.
"""

print("JD characters:", len(jd_text))
print("Resume characters:", len(resume_text))

In [ ]:
# Run the full Keywords Extractor flow for both inputs.
# run_keywords_extractor executes Phase 1 extraction and Phase 2 validation.
jd_state = run_keywords_extractor(jd_text, max_phase1_iter=3, max_phase2_iter=3)
resume_state = run_keywords_extractor(resume_text, max_phase1_iter=3, max_phase2_iter=3)

save_json(OUTPUT_DIR / "end_to_end_jd_state.json", jd_state.model_dump(mode="json"))
save_json(OUTPUT_DIR / "end_to_end_resume_state.json", resume_state.model_dump(mode="json"))

print("JD extractor phase:", jd_state.phase)
print("Resume extractor phase:", resume_state.phase)
print("JD skills extracted:", len(jd_state.datapoints.skills))
print("Resume skills extracted:", len(resume_state.datapoints.skills))

In [ ]:
# Run the Matcher after both documents have validated datapoints.
# This computes both JD -> resume coverage and resume -> JD relevance. Embedding
# similarity is enabled by default to catch semantic equivalents without growing
# a brittle manual alias list; set MATCH_USE_EMBEDDINGS=false to force the
# deterministic fallback path.
match_result = match_resume_to_jd(
    jd_state.datapoints,
    resume_state.datapoints,
    use_embeddings=MATCH_USE_EMBEDDINGS,
    embedding_threshold=MATCH_EMBEDDING_THRESHOLD,
)
save_json(OUTPUT_DIR / "end_to_end_match_result.json", match_result.model_dump(mode="json"))

summary = {
    "overall_score": match_result.overall_score,
    "bidirectional_score": match_result.bidirectional_score,
    "embedding_enabled": match_result.embedding_enabled,
    "embedding_threshold": match_result.embedding_threshold,
    "jd_to_resume": {
        "score": match_result.jd_to_resume.score,
        "matched_skill_count": match_result.jd_to_resume.matched_skill_count,
    },
    "resume_to_jd": {
        "score": match_result.resume_to_jd.score,
        "matched_skill_count": match_result.resume_to_jd.matched_skill_count,
    },
    "precision": match_result.precision,
    "recall": match_result.recall,
    "f1": match_result.f1,
    "required_recall": match_result.required_recall,
    "matched_skills": [
        {
            "jd_skill": match.jd_skill_name,
            "resume_skill": match.resume_skill_name,
            "similarity": match.similarity,
            "match_type": match.match_type,
        }
        for match in match_result.matched_skills
    ],
    "missing_required_skills": [skill.name for skill in match_result.missing_required_skills],
    "extra_resume_skills": [skill.name for skill in match_result.extra_resume_skills],
}

print(json.dumps(summary, indent=2))

In [ ]:
# These assertions make the notebook fail loudly if the end-to-end flow breaks.
assert jd_state.phase == "complete", jd_state.step
assert resume_state.phase == "complete", resume_state.step
assert match_result.jd_skill_count > 0
assert match_result.resume_skill_count > 0
assert match_result.matched_skill_count > 0
assert match_result.jd_to_resume.direction == "jd_to_resume"
assert match_result.resume_to_jd.direction == "resume_to_jd"
assert match_result.jd_to_resume.matched_skill_count > 0
assert match_result.resume_to_jd.matched_skill_count > 0
assert match_result.overall_score == match_result.bidirectional_score

print("End-to-end local workflow succeeded.")